# **Applied Data Science Project:** *Apartment Rent Predictor*
## Name: Aiden Zeigler

---

## Question I want to answer:
### - *"What factors are most important in determining the price of an apartment near Golden, Colorado?"*
### - *"What determines if a new apartment listing is undervalued or overvalued?"*

## **Step 1:** *Import required libraries*

In [201]:
# Supress warnings
import warnings
warnings.filterwarnings("ignore", category=Warning)
warnings.simplefilter(action='ignore', category=FutureWarning)

# Libraries
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, MinMaxScaler
# from sklearn.ensemble import RandomForestClassifier | Which model am I using?
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix
import matplotlib.pyplot as plt
import numpy as np
from sklearn.model_selection import GridSearchCV
import seaborn as sns

# For API
import requests
import os
import json
from pathlib import Path

# For Data Transformation
from geopy.distance import geodesic
from sklearn.preprocessing import OneHotEncoder

## **Step 2:** *Read and load dataset*

### **SOURCE:** RentCast Rental Listings API
*Link to website:* https://developers.rentcast.io/reference/rental-listings-long-term  
API Key **(not required if listings.csv is downloaded)** is stored in *RENTCAST_API_KEY.txt*

---

We will set our focus on listings not in a specific city (or cities), but rather in a certain radius from the Colorado School of Mines campus. Therefore, we aren't capturing specific Golden, Wheatridge, or Lakewood listings, but instead any that are close enough to the school.  

The Colorado School of Mines is located at *1500 Illinois St, Golden, CO 80401*, or coordinates *(39.751022, -105.222580)*

---

**NOTE:** Will revisit the possibly of web scraping rental listing websites if time allows.
Many popular rental listing websites have scrict "anti-bot" rules that prevent me from preforming my own scrapes.

In [202]:
# RentCast has subscriptions in place for those who'd like to use their APIs.
# Therefore, if we already have the data saved, we don't want to send a request.
if Path("listings.csv").is_file():
    df = pd.read_csv("listings.csv")
    
else:
    # API URL
    url = "https://api.rentcast.io/v1/listings/rental/long-term"

    # Read API Key from file
    with open("RENTCAST_API_KEY.txt") as file:
        API_KEY = file.read().strip()

    # Get a response
    response = requests.get(
        url,
        headers={
            "X-Api-Key": API_KEY,
            "Accept": "application.json"
        },
        params={
            "latitude": "39.751022",        # Latitude of Mines
            "longitude": "-105.222580",     # Longitude of Mines
            "radius": "50",                 # Search for closest listings in a 50 mile radius of Mines
            "status": "Active",             # We want active, current listings
            "limit": "500",                 # Limit to 500 listings (default)
        },
        timeout=30
    )

    # Parse JSON
    listings = response.json()

    # Define fields we want to keep
    fields = ["id", "formattedAddress", "city", "state", "zipCode",
              "price", "latitude", "longitude", "bedrooms", 
              "bathrooms", "squareFootage", "propertyType",
              "status", "listedDate", "daysOnMarket"]

    # Read to pandas DataFrame
    df = pd.DataFrame([
        {field: listing.get(field) for field in fields}
        for listing in listings
    ])
    
    # Save the data to a .csv for future use
    df.to_csv("listings.csv", index=False)
    
# Print first few rows
df.head()

,id,formattedAddress,city,state,zipCode,price,latitude,longitude,bedrooms,bathrooms,squareFootage,propertyType,status,listedDate,daysOnMarket
0,"3233-Eliot-St,-Apt-211,-Denver,-CO-80211","3233 Eliot St, Apt 211, Denver, CO 80211",Denver,CO,80211,1365,39.762531,-105.024109,1,1.0,415.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
1,"19672-E-49th-Ave,-Denver,-CO-80249","19672 E 49th Ave, Denver, CO 80249",Denver,CO,80249,2827,39.785152,-104.760556,3,2.5,1845.0,Single Family,Active,2026-10-15T00:00:00.000Z,1
2,"1109-N-Marion-St,-Apt-3,-Denver,-CO-80218","1109 N Marion St, Apt 3, Denver, CO 80218",Denver,CO,80218,1070,39.734043,-104.972092,1,1.0,425.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
3,"2920-W-32nd-Ave,-Apt-112,-Denver,-CO-80211","2920 W 32nd Ave, Apt 112, Denver, CO 80211",Denver,CO,80211,1285,39.761635,-105.024200,1,1.0,500.0,Apartment,Active,2026-10-15T00:00:00.000Z,1
4,"2920-W-32nd-Ave,-Apt-314,-Denver,-CO-80211","2920 W 32nd Ave, Apt 314, Denver, CO 80211",Denver,CO,80211,1485,39.761635,-105.024200,1,1.0,520.0,Apartment,Active,2026-10-15T00:00:00.000Z,1


In [203]:
# Column types and summary statistics
info = df.info()
describe = df.describe()
print(info, describe)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 500 entries, 0 to 499
Data columns (total 15 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   id                500 non-null    object 
 1   formattedAddress  500 non-null    object 
 2   city              500 non-null    object 
 3   state             500 non-null    object 
 4   zipCode           500 non-null    int64  
 5   price             500 non-null    int64  
 6   latitude          500 non-null    float64
 7   longitude         500 non-null    float64
 8   bedrooms          500 non-null    int64  
 9   bathrooms         498 non-null    float64
 10  squareFootage     489 non-null    float64
 11  propertyType      500 non-null    object 
 12  status            500 non-null    object 
 13  listedDate        500 non-null    object 
 14  daysOnMarket      500 non-null    int64  
dtypes: float64(4), int64(4), object(7)
memory usage: 58.7+ KB
None             zipCode        p

## **Step 3:** Clean the Data

---

1. *Missing values:* We drop rows that contain any missing values, as we do not want to estimate square footage or # of bathrooms (Seems tricky and would rather rely on all real data).
2. *Duplicates:* No duplicates, so nothing to handle here.
3. *Outliers:* We will choose *not* to remove outliers, as we want to identify under or overvalued apartments (Could potentially remove a bargain or a rip-off).

In [204]:
# 1. Missing values

# Check the data for missing values
missings = df.isna().sum()
print(f"Columns with missing values: {missings}")

# Drop any columns that contain missing values
df = df.dropna()

# Verify there are no missing values
missings = df.isna().sum().sum()
print(f"Final count of missing values: {missings}")

Columns with missing values: id                   0
formattedAddress     0
city                 0
state                0
zipCode              0
price                0
latitude             0
longitude            0
bedrooms             0
bathrooms            2
squareFootage       11
propertyType         0
status               0
listedDate           0
daysOnMarket         0
dtype: int64
Final count of missing values: 0


In [211]:
# 2. Duplicates

# Check for duplicates
df[df.duplicated()]

# No duplicates, will not handle

,price,latitude,longitude,bedrooms,bathrooms,squareFootage,daysOnMarket,distanceMines,listedMonth,city_Denver,...,zipCode_80236,zipCode_80237,zipCode_80239,zipCode_80246,zipCode_80247,zipCode_80249,propertyType_Apartment,propertyType_Condo,propertyType_Single Family,propertyType_Townhouse
26,1545,1.548107,-1.465141,-0.392975,-0.478118,-0.241762,-0.302899,-1.460155,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
64,4055,-1.399797,0.441750,1.520814,2.450732,1.842538,-0.302899,0.621512,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
67,4055,-1.399797,0.441750,1.520814,2.450732,1.842538,-0.302899,0.621512,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
184,1429,0.663495,-0.555797,0.563919,-0.478118,-0.306658,-0.302899,-0.637506,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
327,890,0.131309,-0.420460,-1.349869,-0.478118,-0.804829,-0.302899,-0.473325,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
430,1199,0.560583,-1.549873,-0.392975,-0.478118,-0.442176,-0.164520,-1.586363,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
437,1263,-0.594070,0.568739,0.563919,0.986307,0.292674,-0.164520,0.562716,0.18431,1.0,...,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0,0.0,0.0
451,995,-1.359293,-1.170205,-0.392975,-0.478118,-0.404002,-0.164520,-0.817485,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
475,799,-0.721841,-0.796657,-0.392975,-0.478118,-0.747568,-0.164520,-0.677328,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


## **Step 5:** Data Engineering

---

1. *Creating features:* We create a new feature, 'distanceMines', that is the distance between Colorado School of Mines and the listing.
2. *Dropping columns:* ID columns such as 'id' or 'formattedDistance' aren't relevant, so we drop them.

In [206]:
# 1. Creating features

# Coordinates of CSM
minesCoordinates = (39.751022, -105.222580)

# Calculate the distance between CSM and the listing (in miles).
df["distanceMines"] = df.apply(
    lambda row: geodesic(
        minesCoordinates,
        (row["latitude"], row["longitude"])
    ).miles,
    axis=1
)

# Print first few rows
df["distanceMines"].head()

0    10.598178
1    24.710734
2    13.392322
3    10.588961
4    10.588961
Name: distanceMines, dtype: float64

In [207]:
# 2. Dropping columns

# Drop unnecessary columns
df = df.drop(columns=["id", "formattedAddress", "state", "status"])

# Print first few rows
df.head()

,city,zipCode,price,latitude,longitude,bedrooms,bathrooms,squareFootage,propertyType,listedDate,daysOnMarket,distanceMines
0,Denver,80211,1365,39.762531,-105.024109,1,1.0,415.0,Apartment,2026-10-15T00:00:00.000Z,1,10.598178
1,Denver,80249,2827,39.785152,-104.760556,3,2.5,1845.0,Single Family,2026-10-15T00:00:00.000Z,1,24.710734
2,Denver,80218,1070,39.734043,-104.972092,1,1.0,425.0,Apartment,2026-10-15T00:00:00.000Z,1,13.392322
3,Denver,80211,1285,39.761635,-105.024200,1,1.0,500.0,Apartment,2026-10-15T00:00:00.000Z,1,10.588961
4,Denver,80211,1485,39.761635,-105.024200,1,1.0,520.0,Apartment,2026-10-15T00:00:00.000Z,1,10.588961


## **Step 4:** Data Transformation

---

1. *Date processing:* We convert our 'listedDate' to datetime and only keep the month, as the year is the same for all listings.
3. *Standardize:* We standardize the numerical columns to prepare for the model.
2. *One-hot encode:* We one-hot encode the catagorical columns to prepare for the model.

In [208]:
# 1. Date processing

# Convert 'listedDate' to datetime
df["listedDate"] = pd.to_datetime(df["listedDate"])

# Extract month only
df["listedMonth"] = df["listedDate"].dt.month

# Remove 'listedDate'
df = df.drop(columns=["listedDate"])

# Print first few rows and unique months
head = df["listedMonth"].head()
unique = df["listedMonth"].unique()
print(head, unique)
df.info()

0    10
1    10
2    10
3    10
4    10
Name: listedMonth, dtype: int32 [10  9  8]
<class 'pandas.core.frame.DataFrame'>
Index: 487 entries, 0 to 499
Data columns (total 12 columns):
 #   Column         Non-Null Count  Dtype  
---  ------         --------------  -----  
 0   city           487 non-null    object 
 1   zipCode        487 non-null    int64  
 2   price          487 non-null    int64  
 3   latitude       487 non-null    float64
 4   longitude      487 non-null    float64
 5   bedrooms       487 non-null    int64  
 6   bathrooms      487 non-null    float64
 7   squareFootage  487 non-null    float64
 8   propertyType   487 non-null    object 
 9   daysOnMarket   487 non-null    int64  
 10  distanceMines  487 non-null    float64
 11  listedMonth    487 non-null    int32  
dtypes: float64(5), int32(1), int64(4), object(2)
memory usage: 47.6+ KB


In [209]:
# 2. Standardize

# Define numerical columns
numerical = ["latitude", "longitude", "bedrooms", "bathrooms", "squareFootage",
             "daysOnMarket", "distanceMines", "listedMonth"]

# Standardize
scaler = StandardScaler()
df[numerical] = scaler.fit_transform(df[numerical])

# Print first few rows
df[numerical].head()

,latitude,longitude,bedrooms,bathrooms,squareFootage,daysOnMarket,distanceMines,listedMonth
0,1.149922,-1.107341,-0.392975,-0.478118,-0.795285,-0.302899,-1.159280,0.18431
1,1.799287,3.707533,1.520814,1.718519,1.934156,-0.302899,3.486304,0.18431
2,0.332138,-0.157037,-0.392975,-0.478118,-0.776198,-0.302899,-0.239501,0.18431
3,1.124202,-1.109003,-0.392975,-0.478118,-0.633046,-0.302899,-1.162314,0.18431
4,1.124202,-1.109003,-0.392975,-0.478118,-0.594872,-0.302899,-1.162314,0.18431


In [210]:
# 3. One-hot encode

# Define categorical columns
categorical = ["city", "zipCode", "propertyType"]

# One-hot encode
encoder = OneHotEncoder(
    sparse_output=False,
    handle_unknown="ignore"
)

# Encode
encoded = pd.DataFrame(
    encoder.fit_transform(df[categorical]),
    columns=encoder.get_feature_names_out(categorical),
    index=df.index
)

# Replace original categorical columns with encoded columns
df = pd.concat(
    [df.drop(columns=categorical), encoded],
    axis=1
)

# Print first few rows
df.head()


,price,latitude,longitude,bedrooms,bathrooms,squareFootage,daysOnMarket,distanceMines,listedMonth,city_Denver,...,zipCode_80236,zipCode_80237,zipCode_80239,zipCode_80246,zipCode_80247,zipCode_80249,propertyType_Apartment,propertyType_Condo,propertyType_Single Family,propertyType_Townhouse
0,1365,1.149922,-1.107341,-0.392975,-0.478118,-0.795285,-0.302899,-1.159280,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
1,2827,1.799287,3.707533,1.520814,1.718519,1.934156,-0.302899,3.486304,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,1.0,0.0
2,1070,0.332138,-0.157037,-0.392975,-0.478118,-0.776198,-0.302899,-0.239501,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
3,1285,1.124202,-1.109003,-0.392975,-0.478118,-0.633046,-0.302899,-1.162314,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0
4,1485,1.124202,-1.109003,-0.392975,-0.478118,-0.594872,-0.302899,-1.162314,0.18431,1.0,...,0.0,0.0,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0.0


# ***STOP:*** *End of Data Cleaning and Preprocessing*

## **Step 6:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 7:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 8:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 9:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 10:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 11:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 12:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 13:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 14:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 15:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 16:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 17:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 18:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 19:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 20:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 21:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 22:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 23:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

## **Step 24:** _

---

1. *Step1:* 
2. *Step2:* 
3. *Step3:* 
4. *Step4:*

# **CONCLUSION:**